# AI-Driven Spare-Parts Inventory Optimization at Honeywell Aerospace
## Multi-Objective Optimization Analysis

**Student Name:** Christelle Jean   
**Professor:** Alex Shaw   
**Target Company/Industry:** Honeywell Aerospace / Aerospace Supply Chain Management  
**Problem:** Selecting safety-stock levels and reorder quantities for critical aerospace spare parts.  
**Objectives:** Minimize expected annual inventory cost vs. maximize immediate part availability.         
**Date:** 11/14/2026


## Project Overview

This project examines how Honeywell Aerospace could balance inventory cost with the availability of critical spare parts. Keeping more inventory can prevent maintenance delays, but it also ties up cash and increases storage and obsolescence costs. I will test different combinations of safety-stock levels and reorder quantities and use Pareto analysis to identify the efficient policies. The goal is to show which inventory choices offer meaningful tradeoffs and which choices are inefficient.

# Section 1: Data Acquisition

## Synthetic Data Approach

Honeywell does not publish its part-level demand, inventory, lead-time, or cost records. I therefore created a synthetic dataset for one representative high-criticality aerospace spare part. The dataset compares 100 policies formed from ten safety-stock levels and ten reorder quantities.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create the required project folders if they do not already exist.
DATA_DIR = Path("data")
IMAGE_DIR = Path("images")
DATA_DIR.mkdir(exist_ok=True)
IMAGE_DIR.mkdir(exist_ok=True)

# A fixed seed makes the synthetic results reproducible.
RANDOM_SEED = 20261115
rng = np.random.default_rng(RANDOM_SEED)

# Decision-variable choices
safety_stock_levels = range(5, 51, 5)
reorder_quantities = range(10, 101, 10)

# Assumptions for one representative critical aerospace spare part
annual_demand = 120
demand_variability_cv = 0.75
supplier_lead_time_days = 45
part_criticality = "High"
unit_cost = 12_000
annual_holding_rate = 0.20
ordering_cost_per_order = 1_500
stockout_penalty_per_unit = 6_000
emergency_premium_per_unit = 2_000
obsolescence_rate = 0.03
storage_per_unit = 1.8
downtime_hours_per_stockout = 12

records = []
policy_number = 1

for safety_stock in safety_stock_levels:
    for reorder_quantity in reorder_quantities:
        # Expected demand during supplier lead time plus safety stock
        lead_time_demand = annual_demand * supplier_lead_time_days / 365
        reorder_point = lead_time_demand + safety_stock

        # Average inventory under a basic continuous-review policy
        average_inventory = safety_stock + (reorder_quantity / 2)
        annual_orders = annual_demand / reorder_quantity

        # Safety stock has the strongest effect on availability.
        # Reorder quantity has a smaller effect, and both have diminishing returns.
        random_variation = rng.normal(0, 0.25)
        fill_rate = (
            82
            + 15.5 * (1 - np.exp(-safety_stock / 18))
            + 2.0 * (1 - np.exp(-reorder_quantity / 45))
            + random_variation
        )
        fill_rate = float(np.clip(fill_rate, 0, 99.6))

        expected_stockouts = annual_demand * (1 - fill_rate / 100)

        # Annual inventory-policy costs
        holding_cost = (
            average_inventory * unit_cost * annual_holding_rate
        )
        ordering_cost = annual_orders * ordering_cost_per_order
        shortage_cost = expected_stockouts * stockout_penalty_per_unit
        emergency_order_cost = (
            expected_stockouts * emergency_premium_per_unit
        )
        obsolescence_cost = (
            average_inventory * unit_cost * obsolescence_rate
        )

        annual_inventory_cost = (
            holding_cost
            + ordering_cost
            + shortage_cost
            + emergency_order_cost
            + obsolescence_cost
        )

        records.append(
            {
                "policy_id": f"P{policy_number:03d}",
                "safety_stock_units": safety_stock,
                "reorder_quantity_units": reorder_quantity,
                "reorder_point_units": round(reorder_point, 1),
                "annual_demand_units": annual_demand,
                "demand_variability_cv": demand_variability_cv,
                "supplier_lead_time_days": supplier_lead_time_days,
                "part_criticality": part_criticality,
                "unit_cost_usd": unit_cost,
                "annual_holding_rate": annual_holding_rate,
                "ordering_cost_per_order_usd": ordering_cost_per_order,
                "stockout_penalty_per_unit_usd": stockout_penalty_per_unit,
                "emergency_premium_per_unit_usd": emergency_premium_per_unit,
                "obsolescence_rate": obsolescence_rate,
                "average_inventory_units": round(average_inventory, 1),
                "annual_orders": round(annual_orders, 2),
                "expected_stockout_units": round(expected_stockouts, 2),
                "holding_cost_usd": round(holding_cost, 2),
                "ordering_cost_usd": round(ordering_cost, 2),
                "shortage_cost_usd": round(shortage_cost, 2),
                "emergency_order_cost_usd": round(
                    emergency_order_cost, 2
                ),
                "obsolescence_cost_usd": round(
                    obsolescence_cost, 2
                ),
                "annual_inventory_cost_usd": round(
                    annual_inventory_cost, 2
                ),
                "immediate_fill_rate_pct": round(fill_rate, 2),
                "storage_used_cubic_feet": round(
                    average_inventory * storage_per_unit, 1
                ),
                "expected_downtime_hours": round(
                    expected_stockouts * downtime_hours_per_stockout, 1
                ),
            }
        )

        policy_number += 1

df = pd.DataFrame(records)

# Save the dataset in the repository's data folder.
data_path = DATA_DIR / "synthetic_inventory_policies.csv"
df.to_csv(data_path, index=False)

# Basic generation checks
assert len(df) == 100
assert df[
    ["safety_stock_units", "reorder_quantity_units"]
].duplicated().sum() == 0
assert df["immediate_fill_rate_pct"].between(0, 100).all()

print("Synthetic dataset created successfully.")
print(f"File: {data_path}")
print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")

df.head()

Synthetic dataset created successfully.
File: data\synthetic_inventory_policies.csv
Rows: 100
Columns: 26


,policy_id,safety_stock_units,reorder_quantity_units,reorder_point_units,annual_demand_units,demand_variability_cv,supplier_lead_time_days,part_criticality,unit_cost_usd,annual_holding_rate,...,expected_stockout_units,holding_cost_usd,ordering_cost_usd,shortage_cost_usd,emergency_order_cost_usd,obsolescence_cost_usd,annual_inventory_cost_usd,immediate_fill_rate_pct,storage_used_cubic_feet,expected_downtime_hours
0,P001,5,10,19.8,120,0.75,45,High,12000,0.2,...,16.61,24000.0,18000.0,99632.54,33210.85,3600.0,178443.39,86.16,18.0,199.3
1,P002,5,20,19.8,120,0.75,45,High,12000,0.2,...,16.29,36000.0,9000.0,97752.22,32584.07,5400.0,180736.29,86.42,27.0,195.5
2,P003,5,30,19.8,120,0.75,45,High,12000,0.2,...,16.35,48000.0,6000.0,98120.63,32706.88,7200.0,192027.51,86.37,36.0,196.2
3,P004,5,40,19.8,120,0.75,45,High,12000,0.2,...,15.83,60000.0,4500.0,94973.47,31657.82,9000.0,200131.29,86.81,45.0,189.9
4,P005,5,50,19.8,120,0.75,45,High,12000,0.2,...,15.72,72000.0,3600.0,94321.50,31440.50,10800.0,212162.01,86.90,54.0,188.6


# Section 2: Data Exploration

# Section 3: Pareto Frontier Calculation

# Section 4: Strategic Analysis

# Section 5: Visualization